# Plotly Tutorial for Astronomy Data

This notebook introduces **Plotly** for interactive astronomy data visualization.

We will use a synthetic Gaia-like star-cluster catalogue with:

- right ascension and declination,
- parallax and distance,
- proper motions,
- colour and magnitude,
- stellar mass,
- group labels,
- disk classification,
- 3D Cartesian positions,
- simple orbit-like motion tracks.

Plotly is especially useful when you want interactive plots with hover labels, zooming, filtering, animation, and 3D visualization.

## 1. Why use Plotly in astronomy?

Plotly is useful for:

- exploring large catalogues interactively,
- inspecting individual sources through hover labels,
- plotting colour–magnitude diagrams,
- visualizing sky positions,
- making interactive 3D maps,
- comparing stellar groups,
- preparing interactive teaching notebooks,
- exporting figures as HTML files.

For final journal figures, Matplotlib is still often preferred, but Plotly is excellent for exploration, teaching, and presentations.

## 2. Install and import packages

## If Plotly figures do not display

This notebook uses a helper function called `show_fig(fig)`.

Each figure is both:
1. displayed inline using Plotly, and
2. saved as an HTML file, such as `plotly_astronomy_plot_01.html`.

If an interactive figure does not appear:
- run the import/renderer setup cell again,
- try changing `setup_plotly_renderer("iframe")` to one of:
  - `"notebook_connected"`
  - `"notebook"`
  - `"jupyterlab"`
  - `"vscode"`
  - `"colab"`
  - `"browser"`
- open the saved `.html` file in your browser.

In JupyterLab, you may need:

```bash
pip install jupyterlab_widgets ipywidgets
```

In [13]:
import plotly.io as pio
pio.renderers.default = 'colab'
fig.show()

In [16]:
# Quick Plotly display test
test_fig = px.scatter(
    x=[1, 2, 3],
    y=[1, 4, 9],
    title="Plotly display test"
)
pio.renderers.default = 'colab'
pio.renderers.default = 'colab'
fig.show()
#show_fig(test_fig, filename="plotly_display_test.html")

In [17]:
# Uncomment if needed:
# %pip install -U numpy pandas plotly scipy

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

np.random.seed(42)

# Robust Plotly renderer setup
#
# Different environments need different renderers:
# - Classic Jupyter Notebook: "notebook" or "notebook_connected"
# - JupyterLab / VS Code: often "iframe" is safest
# - Google Colab: "colab"
# - Offline fallback: save/display HTML
#
# You can manually change this to: "notebook", "notebook_connected",
# "jupyterlab", "vscode", "colab", "iframe", or "browser".
import plotly.io as pio
import os, sys
from IPython.display import HTML, display

def setup_plotly_renderer(preferred=None):
    if preferred is not None:
        pio.renderers.default = preferred
        return preferred

    # Colab detection
    if "google.colab" in sys.modules:
        pio.renderers.default = "colab"
        return "colab"

    # VS Code notebooks often work well with vscode, otherwise iframe
    if "VSCODE_PID" in os.environ:
        pio.renderers.default = "vscode"
        return "vscode"

    # iframe is usually the most robust offline option in Jupyter/JupyterLab
    pio.renderers.default = "iframe"
    return "iframe"

ACTIVE_RENDERER = setup_plotly_renderer()
print("Plotly renderer:", ACTIVE_RENDERER)

def show_fig(fig, filename=None, width="100%", height="650px"):
    """
    Robustly show a Plotly figure.

    1. Try fig.show() using the active renderer.
    2. Also save the figure as HTML.
    3. If inline rendering fails in your environment, open the HTML file.
    """
    if filename is None:
        filename = "plotly_figure.html"

    fig.write_html(filename, include_plotlyjs="cdn", full_html=True)

    try:
        show_fig(fig, filename="plotly_astronomy_plot_01.html")
    except Exception as err:
        print("fig.show() failed:", repr(err))
        print(f"Open the saved HTML file instead: {filename}")
        display(HTML(f'<iframe src="{filename}" width="{width}" height="{height}" style="border:0;"></iframe>'))

    return filename

Plotly renderer: colab


## 3. Create a synthetic Gaia-like astronomy catalogue

In [18]:
def make_group(name, n, ra0, dec0, dist_pc, pmra0, pmdec0, age_myr, disk_fraction):
    ra = np.random.normal(ra0, 0.35, n)
    dec = np.random.normal(dec0, 0.25, n)

    distance = np.random.normal(dist_pc, 12, n)
    parallax = 1000.0 / distance + np.random.normal(0, 0.08, n)

    pmra = np.random.normal(pmra0, 0.45, n)
    pmdec = np.random.normal(pmdec0, 0.45, n)

    mass = np.random.lognormal(mean=np.log(0.45), sigma=0.55, size=n)
    mass = np.clip(mass, 0.08, 4.0)

    bp_rp = 0.65 + 1.7 * (1 / np.sqrt(mass)) + np.random.normal(0, 0.12, n)
    abs_g = 3.2 + 4.2 * bp_rp - 1.1 * np.log10(mass + 0.05) + 0.25 * age_myr
    g_mag = abs_g + 5 * np.log10(distance / 10) + np.random.normal(0, 0.18, n)

    has_disk = np.random.random(n) < disk_fraction
    disk_class = np.where(has_disk, "disk-bearing", "diskless")

    # Approximate heliocentric Cartesian coordinates in pc.
    # These are simple teaching coordinates, not a rigorous astrometric transform.
    ra_rad = np.deg2rad(ra)
    dec_rad = np.deg2rad(dec)
    x = distance * np.cos(dec_rad) * np.cos(ra_rad)
    y = distance * np.cos(dec_rad) * np.sin(ra_rad)
    z = distance * np.sin(dec_rad)

    return pd.DataFrame({
        "source_id": [f"{name.replace(' ', '_')}_{i:04d}" for i in range(n)],
        "group": name,
        "ra_deg": ra,
        "dec_deg": dec,
        "distance_pc": distance,
        "parallax_mas": parallax,
        "pmra_masyr": pmra,
        "pmdec_masyr": pmdec,
        "pm_total_masyr": np.sqrt(pmra**2 + pmdec**2),
        "age_myr": age_myr,
        "mass_msun": mass,
        "bp_rp": bp_rp,
        "g_mag": g_mag,
        "absolute_g": abs_g,
        "disk_class": disk_class,
        "x_pc": x,
        "y_pc": y,
        "z_pc": z,
    })

df = pd.concat([
    make_group("Cluster A", 180, 52.2, 31.3, 295, 5.0, -7.8, 1.5, 0.55),
    make_group("Cluster B", 240, 56.1, 32.1, 320, 4.0, -6.2, 3.0, 0.30),
    make_group("Association C", 160, 59.5, 34.2, 340, 2.8, -5.2, 6.0, 0.12),
], ignore_index=True)

display(df.head())
print(df.shape)

,source_id,group,ra_deg,dec_deg,distance_pc,parallax_mas,pmra_masyr,pmdec_masyr,pm_total_masyr,age_myr,mass_msun,bp_rp,g_mag,absolute_g,disk_class,x_pc,y_pc,z_pc
0,Cluster_A_0000,Cluster A,52.373850,31.456417,301.232158,3.386554,5.138511,-7.634097,9.202376,1.5,0.371478,3.506218,26.227688,18.713862,diskless,156.877199,203.516980,157.197952
1,Cluster_A_0001,Cluster A,52.151607,31.085711,313.392867,3.100507,4.230424,-7.977002,9.029344,1.5,0.371354,3.448804,26.164534,18.472863,diskless,164.676175,211.929397,161.810931
2,Cluster_A_0002,Cluster A,52.426691,31.032277,293.694878,3.447279,4.393317,-7.787065,8.940895,1.5,0.377039,3.483223,25.983088,18.611022,disk-bearing,153.456471,199.459432,151.405839
3,Cluster_A_0003,Cluster A,52.733060,31.420618,299.820541,3.450654,5.334469,-7.224697,8.980690,1.5,1.410161,1.971096,18.823639,11.672762,disk-bearing,154.928203,203.615959,156.301470
4,Cluster_A_0004,Cluster A,52.118046,31.244134,303.281728,3.099533,5.076889,-7.714005,9.234754,1.5,0.555191,2.951861,23.693912,16.212736,disk-bearing,159.216786,204.655917,157.307905


(580, 18)


## 4. Interactive sky plot

A sky plot is often the first view of a cluster or star-forming region.

Plotly lets you hover over individual stars and inspect metadata.

In [21]:
fig = px.scatter(
    df,
    x="ra_deg",
    y="dec_deg",
    color="group",
    symbol="disk_class",
    hover_data=["source_id", "distance_pc", "parallax_mas", "pmra_masyr", "pmdec_masyr", "g_mag"],
    title="Interactive sky distribution",
    labels={
        "ra_deg": "Right Ascension [deg]",
        "dec_deg": "Declination [deg]",
    },
)

# Astronomers usually plot RA increasing to the left.
fig.update_xaxes(autorange="reversed")
fig.update_layout(width=850, height=550)
pio.renderers.default = 'colab'
fig.show()
#show_fig(fig, filename="plotly_astronomy_plot_02.html")


### Interpretation

Use this plot to inspect whether the groups occupy different sky regions.  
Spatial clustering alone is not enough for membership selection: proper motion, parallax, photometry, and youth indicators are also needed.

## 5. Interactive proper-motion diagram

In [24]:
fig = px.scatter(
    df,
    x="pmra_masyr",
    y="pmdec_masyr",
    color="group",
    symbol="disk_class",
    size="mass_msun",
    hover_data=["source_id", "ra_deg", "dec_deg", "distance_pc", "g_mag"],
    title="Proper-motion diagram",
    labels={
        "pmra_masyr": "Proper motion RA* [mas/yr]",
        "pmdec_masyr": "Proper motion Dec [mas/yr]",
        "mass_msun": "Mass [M_sun]",
    },
)
fig.update_layout(width=850, height=550)
pio.renderers.default = 'colab'
fig.show()
#show_fig(fig, filename="plotly_astronomy_plot_03.html")


### Interpretation

Groups that are physically related often cluster in proper-motion space.  
Hovering helps identify outliers or possible contaminants.

## 6. Interactive colour–magnitude diagram

In [27]:
fig = px.scatter(
    df,
    x="bp_rp",
    y="absolute_g",
    color="group",
    symbol="disk_class",
    size="mass_msun",
    hover_data=["source_id", "distance_pc", "parallax_mas", "age_myr", "g_mag"],
    title="Interactive colour–magnitude diagram",
    labels={
        "bp_rp": "BP - RP [mag]",
        "absolute_g": "Absolute G [mag]",
        "mass_msun": "Mass [M_sun]",
    },
)

# Magnitudes are inverted: brighter stars at the top.
fig.update_yaxes(autorange="reversed")
fig.update_layout(width=850, height=650)
pio.renderers.default = 'colab'
fig.show()


### Interpretation

A colour–magnitude diagram can reveal:

- age differences,
- reddening/extinction,
- binaries,
- field-star contamination,
- pre-main-sequence evolution.

In real work, compare the CMD with theoretical isochrones instead of relying only on visual trends.

## 7. Histogram and marginal distributions

In [29]:
fig = px.histogram(
    df,
    x="distance_pc",
    color="group",
    marginal="box",
    nbins=40,
    opacity=0.65,
    barmode="overlay",
    title="Distance distributions with marginal box plots",
    labels={"distance_pc": "Distance [pc]"},
)
fig.update_layout(width=850, height=550)
pio.renderers.default = 'colab'
fig.show()



## 8. Violin plot of stellar mass by group

In [31]:
fig = px.violin(
    df,
    x="group",
    y="mass_msun",
    color="group",
    box=True,
    points="all",
    hover_data=["source_id", "bp_rp", "absolute_g"],
    title="Mass distribution by group",
    labels={"mass_msun": "Mass [M_sun]"},
)
fig.update_layout(width=850, height=550, showlegend=False)
pio.renderers.default = 'colab'
fig.show()


### Caution

Mass distributions are highly sensitive to selection effects, completeness, extinction, and the adopted age model.

## 9. Disk fraction bar chart

In [32]:
disk_fraction = (
    df.assign(has_disk=df["disk_class"].eq("disk-bearing"))
      .groupby("group", as_index=False)
      .agg(disk_fraction=("has_disk", "mean"), n=("has_disk", "size"))
)

fig = px.bar(
    disk_fraction,
    x="group",
    y="disk_fraction",
    text="n",
    title="Disk fraction by group",
    labels={"disk_fraction": "Disk fraction", "group": "Group"},
)
fig.update_yaxes(range=[0, 1])
fig.update_layout(width=750, height=500)
pio.renderers.default = 'colab'
fig.show()

display(disk_fraction)

,group,disk_fraction,n
0,Association C,0.106250,160
1,Cluster A,0.488889,180
2,Cluster B,0.283333,240


### Interpretation

Disk fraction can be used as a rough youth indicator.  
Younger clusters often have larger disk fractions, but disk classification depends on wavelength coverage and survey depth.

## 10. Interactive 3D map of stellar positions

In [34]:
fig = px.scatter_3d(
    df,
    x="x_pc",
    y="y_pc",
    z="z_pc",
    color="group",
    symbol="disk_class",
    size="mass_msun",
    hover_data=["source_id", "distance_pc", "parallax_mas", "pm_total_masyr"],
    title="Interactive 3D stellar positions",
    labels={
        "x_pc": "X [pc]",
        "y_pc": "Y [pc]",
        "z_pc": "Z [pc]",
    },
)
fig.update_layout(width=900, height=700)
pio.renderers.default = 'colab'
fig.show()



### Interpretation

3D plots are helpful for teaching and exploration.  
For precise science, use rigorous coordinate transformations with `astropy.coordinates`.

## 11. Proper-motion vector plot with Plotly graph objects

In [35]:
# Use group centroids for a cleaner vector plot
centroids = (
    df.groupby("group", as_index=False)
      .agg(
          ra_deg=("ra_deg", "median"),
          dec_deg=("dec_deg", "median"),
          pmra_masyr=("pmra_masyr", "median"),
          pmdec_masyr=("pmdec_masyr", "median"),
          age_myr=("age_myr", "median"),
          n=("source_id", "size")
      )
)

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=centroids["ra_deg"],
    y=centroids["dec_deg"],
    mode="markers+text",
    text=centroids["group"],
    textposition="top center",
    marker=dict(
        size=centroids["n"] / 8,
        color=centroids["age_myr"],
        colorscale="Viridis",
        showscale=True,
        colorbar=dict(title="Age [Myr]"),
    ),
    hovertemplate=(
        "Group=%{text}<br>"
        "RA=%{x:.2f} deg<br>"
        "Dec=%{y:.2f} deg<br>"
        "<extra></extra>"
    )
))

# Add motion arrows as line segments
scale = 0.20
for _, row in centroids.iterrows():
    x0 = row["ra_deg"]
    y0 = row["dec_deg"]
    x1 = x0 + row["pmra_masyr"] * scale
    y1 = y0 + row["pmdec_masyr"] * scale

    fig.add_trace(go.Scatter(
        x=[x0, x1],
        y=[y0, y1],
        mode="lines",
        line=dict(width=3),
        showlegend=False,
        hoverinfo="skip",
    ))

fig.update_xaxes(autorange="reversed", title="Right Ascension [deg]")
fig.update_yaxes(title="Declination [deg]")
fig.update_layout(
    title="Group centroid proper-motion vectors",
    width=850,
    height=600,
)
pio.renderers.default = 'colab'
fig.show()



### Interpretation

This plot shows the median motion of each group on the plane of the sky.  
It does not include radial velocity, so it is not a full 3D velocity analysis.

## 12. Animated traceback-style toy model

In [37]:
# Toy motion model:
# 1 km/s ≈ 1.02 pc/Myr.
# Proper motions are not directly km/s here, so this is a teaching animation,
# not a physical orbit integration.

times = np.linspace(0, -5, 26)  # Myr, present to 5 Myr ago

frames = []
animated_rows = []

for t in times:
    temp = centroids.copy()
    # Artificial scale for visualization only
    temp["time_myr"] = t
    temp["ra_trace"] = temp["ra_deg"] + temp["pmra_masyr"] * 0.03 * t
    temp["dec_trace"] = temp["dec_deg"] + temp["pmdec_masyr"] * 0.03 * t
    animated_rows.append(temp)

anim_df = pd.concat(animated_rows, ignore_index=True)

fig = px.scatter(
    anim_df,
    x="ra_trace",
    y="dec_trace",
    animation_frame="time_myr",
    color="group",
    size="n",
    hover_data=["group", "age_myr", "pmra_masyr", "pmdec_masyr"],
    title="Toy traceback animation using plane-of-sky motion",
    labels={
        "ra_trace": "RA-like coordinate [deg]",
        "dec_trace": "Dec-like coordinate [deg]",
        "time_myr": "Time [Myr]",
    },
)
fig.update_xaxes(autorange="reversed")
fig.update_layout(width=850, height=600)
pio.renderers.default = 'colab'
fig.show()



### Caution

This is a teaching animation, not a dynamical orbit model.  
For real traceback work, use radial velocities, distances, uncertainties, and a Galactic potential with tools such as `astropy` and `galpy`.

## 13. Interactive matrix plot

In [38]:
fig = px.scatter_matrix(
    df.sample(400, random_state=1),
    dimensions=["distance_pc", "pmra_masyr", "pmdec_masyr", "bp_rp", "absolute_g"],
    color="group",
    hover_data=["source_id", "disk_class"],
    title="Interactive scatter-matrix plot",
)
fig.update_traces(diagonal_visible=False, showupperhalf=False)
fig.update_layout(width=950, height=850)
pio.renderers.default = 'colab'
fig.show()



### Interpretation

Scatter-matrix plots help identify which variables separate groups.  
They are useful for membership analysis and data-cleaning diagnostics.

## 14. Export Plotly figures to HTML

In [40]:
fig = px.scatter(
    df,
    x="bp_rp",
    y="absolute_g",
    color="group",
    symbol="disk_class",
    hover_data=["source_id", "distance_pc", "parallax_mas"],
    title="Exported interactive CMD",
)
fig.update_yaxes(autorange="reversed")

fig.write_html("interactive_cmd_plotly.html")
pio.renderers.default = 'colab'
fig.show()
print("Saved: interactive_cmd_plotly.html")

Saved: interactive_cmd_plotly.html


## 15. Using Plotly with real astronomy data

For real Gaia or VizieR data, the workflow is usually:

```python
from astropy.table import Table
tab = Table.read("my_catalogue.vot", format="votable")
df = tab.to_pandas()
```

Then use Plotly Express:

```python
px.scatter(df, x="bp_rp", y="phot_g_mean_mag", color="group")
```

Recommended real-data checks:

- remove rows with missing parallax/proper motion,
- check RUWE or astrometric quality,
- handle extinction and reddening,
- correct parallax zero-point if needed,
- avoid simple parallax inversion for noisy parallaxes,
- use homogeneous membership selection when comparing clusters.

# Exercises

## Exercise 1: Interactive parallax histogram
Make a Plotly histogram of `parallax_mas` grouped by `group`.  
Add a marginal box plot.

## Exercise 2: Disk-bearing stars on the CMD
Make an interactive CMD where colour represents `disk_class` and symbol represents `group`.

## Exercise 3: 3D map coloured by age
Make a 3D scatter plot of `x_pc`, `y_pc`, and `z_pc`, coloured by `age_myr`.

## Exercise 4: Select possible members
Select stars with:

```python
4.0 < pmra_masyr < 6.0
-8.5 < pmdec_masyr < -6.5
```

Plot their sky distribution and CMD.

## Exercise 5: Export an HTML plot
Create a Plotly figure and save it as:

```python
my_astronomy_plot.html
```

## Exercise 6: Challenge
Create an animated CMD where the animation frame is `group`.  
This is not physical time evolution; it is just a way to inspect groups one at a time.

## Exercise solution templates

In [ ]:
# Exercise 1 template
# fig = px.histogram(
#     df,
#     x="parallax_mas",
#     color="group",
#     marginal="box",
#     nbins=40,
#     opacity=0.65,
#     barmode="overlay",
# )
# fig.show()

In [ ]:
# Exercise 4 template
# selected = df[
#     (df["pmra_masyr"] > 4.0) &
#     (df["pmra_masyr"] < 6.0) &
#     (df["pmdec_masyr"] > -8.5) &
#     (df["pmdec_masyr"] < -6.5)
# ]
#
# fig = px.scatter(selected, x="ra_deg", y="dec_deg", color="group", hover_data=["source_id"])
# fig.update_xaxes(autorange="reversed")
# fig.show()
#
# fig = px.scatter(selected, x="bp_rp", y="absolute_g", color="group")
# fig.update_yaxes(autorange="reversed")
# fig.show()

# Final notes

Plotly is excellent for interactive astronomy exploration.

Good practice:

- use hover labels to inspect sources,
- label axes with units,
- invert magnitude axes,
- export interactive HTML for teaching or collaboration,
- avoid over-interpreting toy animations,
- use rigorous coordinate transformations for real 3D work.